# Phase 3 — Benchmark 1 Million Healthcare Records with PySpark

**Goal:** Generate **100,000 synthetic patients**, **2,000 providers**, **1,000,000 encounters**, and **1,000,000 claims**; execute the existing Bronze → Silver → Gold pipeline; compare Spark partition choices; and capture **real measured results**.

We will compare **4, 8, and 16 partitions** on an identical analytics task:

1. Read valid claims from Silver Parquet.
2. Repartition by `service_date`.
3. Aggregate claims and billed cents by date.
4. Write and read results back.
5. Confirm exact count and billed amount reconciliation.

Each run writes a unique output folder. We measure code execution, not Colab startup or package install time.

**Do not claim a performance improvement until your measured numbers are available.** Colab is a single machine, not a production Spark cluster.

## Step 1 — Install PySpark

Use a new Colab runtime, or restart after installing. The warning about the pre-installed `dataproc-spark-connect` package expecting PySpark 4 may appear; the project currently uses Spark 3.5.x.

In [ ]:
%pip -q install "pyspark>=3.5,<4"

## Step 2 — Load the GitHub project

The notebook fetches the latest committed code. If it already exists in this Colab session, `git pull` refreshes it.

In [ ]:
from pathlib import Path
import subprocess, sys
PROJECT = Path("/content/healthcare-lakehouse-data-platform")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/mosesrathan19/healthcare-lakehouse-data-platform.git",
                    str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only"], check=True)
sys.path.insert(0, str(PROJECT))
print("Project ready:", PROJECT)

## Step 3 — Run a smoke test first

Start with the **10K-encounter demo** to verify your Spark runtime before running the larger workload. It writes its measurements under `outputs/benchmark_demo/`.

The full data-pipeline timing includes all quality checks, writes, and Gold calculations.

In [ ]:
import subprocess
def run_benchmark(scale, name):
    cmd = [sys.executable, "-m", "src.benchmark",
           "--scale", scale,
           "--output-dir", f"outputs/{name}",
           "--partitions", "4,8,16"]
    subprocess.run(cmd, cwd=PROJECT, check=True)

run_benchmark("demo", "benchmark_demo")

## Step 4 — Run the 1-million-record workload

**Medium mode:** 1M encounters + 1M claims. The job writes synthetic source Parquet, validates both fact datasets, produces Silver and Gold, then performs three partition comparisons.

Use a Colab runtime with enough free RAM/disk. If the session runs out of memory, reduce the partition comparison or use a more capable runtime. The benchmark does not need Delta Lake or real patient data.

In [ ]:
run_benchmark("medium", "benchmark_1m")

## Step 5 — Inspect the actual measurements

The results JSON captures the run timestamp, Spark environment, data quality metrics, timings, output file sizes, and partition trial results.

**What to look for:** Which partition count finished fastest on *your* runtime? Did all the trials process exactly one million claims and reconcile billed cents?

In [ ]:
import json
import pandas as pd
report_path = PROJECT / "outputs/benchmark_1m/benchmark_results.json"
report = json.loads(report_path.read_text())
print("Source counts:", report["source_counts"])
print("Quality:", report["quality"])
print("End-to-end times (seconds):", report["timings_seconds"])
trials = pd.DataFrame(report["partition_comparison"]["trials"])
display(trials)
assert report["quality"]["encounters_valid"] == 1_000_000
assert report["quality"]["claims_valid"] == 1_000_000
assert all(trials["reconciled"])

## Step 6 — Plot partition performance

Lower elapsed seconds is better for the measured workload. This experiment holds the dataset and query constant, changing only the number of `service_date` partitions, but warm-up and caching effects can still affect outcomes.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.bar(trials["partitions"].astype(str), trials["seconds"])
plt.title("Healthcare 1M claims — Partition comparison (measured)")
plt.xlabel("Repartition count")
plt.ylabel("Elapsed seconds")
plt.tight_layout()
plt.show()
print("Fastest measured trial:", trials.loc[trials["seconds"].idxmin(), "partitions"], "partitions")

## Step 7 — Save the benchmark results for your GitHub portfolio

Download `benchmark_results.json` to keep the actual metrics. The benchmark output dataset is much larger and should **not** be committed to GitHub.

Recommended reporting: use your measured Spark version, CPU/runtime type, input rows, source-generation time, pipeline time, partition timings, and trial consistency. Run the partition comparison more than once before making strong tuning claims.

In [ ]:
from google.colab import files
files.download(str(report_path))

## How to interpret the benchmark

**Measured**: synthetic records generated, full pipeline wall-clock time, same-input partition variants, JSON report and consistency checks.

**Not measured**: cluster scaling, cloud networking, production S3 throughput, concurrency, streaming, task-level shuffle metrics, or memory peak.

**Next phase:** formal data-quality expectations, anomaly handling, and scheduled Airflow orchestration.